# AIMO 3 Submission: SC-TIR with Transformers

**Algorithm: Self-Consistency with Tool-Integrated Reasoning**

```
For each problem:
  1. Generate N=16 solution attempts (with temperature sampling)
  2. Each attempt can write Python code -> we execute it -> feed output back
  3. Repeat code execution up to M=4 times per attempt
  4. Extract \boxed{answer} from each attempt
  5. Majority vote -> final answer
```

**IMPORTANT SETUP (internet is OFF):**
1. Click **"Add Input"** (right sidebar) -> **"Models"** -> search **"qwen2.5 math"**
2. Select **Qwen2.5-Math-7B-Instruct** (by QwenLM) -> Add
3. Also add the competition data: **"Add Input"** -> **"Competitions"** -> **"AIMO Progress Prize 3"**
4. Set accelerator to **GPU** (H100 preferred)

No pip installs needed - uses only pre-installed packages (transformers, torch).

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

import torch
assert torch.cuda.is_available(), "CUDA not available"
print(f"GPU: {torch.cuda.get_device_name(0)}")

import vllm
from vllm import LLM, SamplingParams
USE_VLLM = True
print(f"vLLM version: {vllm.__version__}")

In [ ]:
import os
import re
import signal
import subprocess
import tempfile
from collections import Counter
from contextlib import contextmanager
from dataclasses import dataclass, field
from typing import Optional

import pandas as pd
import torch

IS_SUBMISSION = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))

print(f"Submission mode: {IS_SUBMISSION}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

In [ ]:
# =============================================
# CONFIGURATION
# =============================================

@dataclass
class Config:
    num_samples: int = 16        # N: solution attempts per problem
    num_generations: int = 2     # M: code execution rounds (context-constrained)
    temperature: float = 0.8
    top_p: float = 0.95
    max_tokens: int = 1280       # Per generation step (prompt~800 + 2*1280 < 4096)
    max_model_len: int = 4096    # Qwen2.5-Math-7B hard limit
    code_timeout: int = 10
    model_path: str = "/kaggle/input/models/tantheta/aimo3-qwen-math-7b-sft-grpo/transformers/default/1"

config = Config()

assert os.path.isfile(os.path.join(config.model_path, "config.json")), \
    f"Model not found at {config.model_path}"
print(f"Model path: {config.model_path}")

import glob
print("\nAttached inputs:")
for p in glob.glob("/kaggle/input/*/"):
    print(f"  {p}")

In [ ]:
# =============================================
# PYTHON REPL - Safe code execution sandbox
# =============================================
# When the model writes ```python ... ```, we execute it and
# feed the output back. This is the "Tool" in Tool-Integrated Reasoning.
#
# Safety measures:
# - Runs in a temp directory (no file system access)
# - Timeout kills long-running code (infinite loops)
# - Blocked: subprocess, os.system, eval, exec, __import__

class PythonREPL:
    def __init__(self, timeout: int = 5):
        self.timeout = timeout
    
    @contextmanager
    def _time_limit(self, seconds):
        def handler(*_):
            raise TimeoutError(f"Code timed out after {seconds}s")
        old = signal.signal(signal.SIGALRM, handler)
        signal.alarm(seconds)
        try:
            yield
        finally:
            signal.alarm(0)
            signal.signal(signal.SIGALRM, old)
    
    def execute(self, code: str) -> tuple:
        """Execute Python code. Returns (success: bool, output: str)."""
        # Add common math imports
        preamble = (
            "import math\n"
            "import numpy as np\n"
            "import sympy as sp\n"
            "from sympy import *\n"
            "from fractions import Fraction\n"
            "from itertools import permutations, combinations, product\n"
            "from functools import reduce\n"
        )
        full_code = preamble + code
        
        # Make sure last line prints something
        lines = full_code.strip().split("\n")
        last = lines[-1].split("#")[0].strip()
        if last and "print(" not in last and not last.startswith(("import", "from", "#", "def", "class", "if", "for", "while", "try", "with")):
            lines[-1] = f"print({last})"
            full_code = "\n".join(lines)
        
        with tempfile.TemporaryDirectory() as tmpdir:
            path = os.path.join(tmpdir, "solution.py")
            with open(path, "w") as f:
                f.write(full_code)
            try:
                with self._time_limit(self.timeout):
                    result = subprocess.run(
                        ["python3", path],
                        capture_output=True, text=True, timeout=self.timeout,
                    )
                if result.returncode == 0:
                    return True, result.stdout.strip()[:1000]
                return False, result.stderr.strip()[-500:]
            except (TimeoutError, subprocess.TimeoutExpired):
                return False, "Timed out"
            except Exception as e:
                return False, str(e)[:200]

executor = PythonREPL(config.code_timeout)

# Test it
ok, out = executor.execute("print(sum(range(100)))")
print(f"Test: success={ok}, output={out}")

In [ ]:
# =============================================
# ANSWER EXTRACTION
# =============================================

def extract_boxed(text: str) -> str:
    """Extract answer from \\boxed{...} or \\fbox{...}."""
    for prefix in [r"\boxed", r"\fbox"]:
        idx = text.rfind(prefix)
        if idx < 0:
            continue
        i, depth = idx, 0
        while i < len(text):
            if text[i] == "{":
                depth += 1
            elif text[i] == "}":
                depth -= 1
                if depth == 0:
                    start = text.index("{", idx) + 1
                    return text[start:i].strip()
            i += 1
    return ""


def to_int(text: str) -> int:
    """Parse answer text to integer in range 0-99999."""
    if not text:
        return -1
    
    # Clean up common formatting
    for rm in [r"\text{", "}", r"\mathrm{", "$", ",", " ", "%",
               "square", "units", "degrees", "cm", "meters"]:
        text = text.replace(rm, "")
    
    # Handle fractions
    if "/" in text and text.replace("/", "").replace(".", "").replace("-", "").isdigit():
        try:
            from fractions import Fraction
            val = float(Fraction(text))
            return max(0, min(99999, round(val)))
        except Exception:
            pass
    
    try:
        val = round(float(text))
        return max(0, min(99999, val))
    except Exception:
        return -1


# Test
print(extract_boxed(r"The answer is \boxed{42}"))   # -> "42"
print(to_int("42"))                                  # -> 42
print(to_int("1,234"))                               # -> 1234

In [ ]:
# =============================================
# FIX TOKENIZER CONFIG (version mismatch)
# =============================================
# Our model was saved with transformers 5.x which stores extra_special_tokens
# as a list. Kaggle's transformers expects a dict. Patch the config.

import shutil, json

FIXED_PATH = "/kaggle/working/model_fixed"
SRC = config.model_path

if not os.path.exists(FIXED_PATH):
    os.makedirs(FIXED_PATH, exist_ok=True)
    for fname in os.listdir(SRC):
        src_file = os.path.join(SRC, fname)
        dst_file = os.path.join(FIXED_PATH, fname)
        if fname.endswith(".safetensors") or fname.endswith(".bin"):
            # Symlink large weight files (don't copy 14GB)
            os.symlink(src_file, dst_file)
        else:
            shutil.copy2(src_file, dst_file)

# Patch tokenizer_config.json
tc_path = os.path.join(FIXED_PATH, "tokenizer_config.json")
with open(tc_path) as f:
    tc = json.load(f)
if isinstance(tc.get("extra_special_tokens"), list):
    tc["extra_special_tokens"] = {}
    with open(tc_path, "w") as f:
        json.dump(tc, f, indent=2)
    print("Patched extra_special_tokens (list -> dict)")
else:
    print("tokenizer_config.json already OK")

config.model_path = FIXED_PATH
print(f"Using fixed model path: {config.model_path}")
print("Files:", os.listdir(FIXED_PATH))

In [ ]:
# =============================================
# SC-TIR: The Core Algorithm
# =============================================

BLOCKED_COMMANDS = ["subprocess", "os.system", "__import__", "shutil", "open(", "eval(", "exec("]

def extract_last_code_block(text: str) -> str:
    """Extract the last ```python ... ``` block from text."""
    blocks = re.findall(r"```python\s*(.*?)```", text, re.DOTALL)
    if not blocks:
        return ""
    code = blocks[-1].strip()
    for cmd in BLOCKED_COMMANDS:
        if cmd in code:
            return ""
    return code


def build_prompt(problem: str) -> str:
    """Match the GRPO training prompt exactly."""
    messages = [{"role": "user", "content": (
        "Solve this math problem step by step. "
        "Use Python code when calculations are needed.\n"
        "Put your final numerical answer in \\boxed{}.\n\n"
        f"Problem: {problem}"
    )}]
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def _generate_vllm(candidates: list[str]) -> list[str]:
    sampling = SamplingParams(
        temperature=config.temperature,
        top_p=config.top_p,
        max_tokens=config.max_tokens,
        stop=["```output", "</s>", "<|im_end|>"],
        include_stop_str_in_output=True,
    )
    outputs = llm.generate(candidates, sampling, use_tqdm=False)
    return [o.prompt + o.outputs[0].text for o in outputs]


def solve(problem: str, debug: bool = False) -> int:
    prompt = build_prompt(problem)
    candidates = [prompt] * config.num_samples

    for step in range(config.num_generations):
        candidates = _generate_vllm(candidates)
        new_candidates = []
        any_code = False
        for text in candidates:
            if text.rstrip().endswith("```output"):
                code = extract_last_code_block(text)
                if code:
                    success, result = executor.execute(code)
                    text += f"\n{result}\n```\n"
                    any_code = True
                else:
                    text += "\n\n```\n"
            new_candidates.append(text)
        candidates = new_candidates
        if not any_code:
            break

    answers = []
    for text in candidates:
        boxed = extract_boxed(text)
        if boxed:
            val = to_int(boxed)
            if 0 <= val <= 99999:
                answers.append(val)

    if not answers:
        if debug:
            print("DEBUG: no answers extracted. First candidate last 1500 chars:")
            print(candidates[0][-1500:])
            print("---")
        return 0

    winner = Counter(answers).most_common(1)[0][0]
    print(f"  -> {len(answers)}/{config.num_samples} answers extracted, winner={winner}")
    return winner


# Debug test with verbose output
print("Testing solve on a simple problem...")
test_answer = solve("What is 17 * 23?", debug=True)
print(f"Result: 17*23 = {test_answer} (expected 391)")

In [ ]:
# =============================================
# SC-TIR: The Core Algorithm
# =============================================

BLOCKED_COMMANDS = ["subprocess", "os.system", "__import__", "shutil", "open(", "eval(", "exec("]

def extract_last_code_block(text: str) -> str:
    """Extract the last ```python ... ``` block from text."""
    blocks = re.findall(r"```python\s*(.*?)```", text, re.DOTALL)
    if not blocks:
        return ""
    code = blocks[-1].strip()
    for cmd in BLOCKED_COMMANDS:
        if cmd in code:
            return ""
    return code


def build_prompt(problem: str) -> str:
    """Build the chat prompt for a math problem."""
    messages = [{"role": "user", "content": (
        "Solve the following math problem step by step.\n"
        "Use Python code (inside ```python ... ``` blocks) whenever calculations are needed. "
        "The code will be executed and its output returned to you.\n"
        "The final answer is a non-negative integer between 0 and 99999 (inclusive). "
        "Do NOT reduce modulo 1000 — give the full integer.\n"
        "Put your final answer inside \\boxed{}.\n\n"
        f"Problem: {problem}"
    )}]
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def _generate_vllm(candidates: list[str]) -> list[str]:
    """Generate next step for all candidates using vLLM (fast, batched)."""
    sampling = SamplingParams(
        temperature=config.temperature,
        top_p=config.top_p,
        max_tokens=config.max_tokens,
        stop=["```output", "</s>", "<|im_end|>"],
        include_stop_str_in_output=True,
    )
    outputs = llm.generate(candidates, sampling, use_tqdm=False)
    return [o.prompt + o.outputs[0].text for o in outputs]


def solve(problem: str) -> int:
    """
    Solve a math problem using SC-TIR.

    1. Create N copies of the prompt
    2. Generate solutions (stop at ```output to execute code)
    3. Execute code, append result, continue generating
    4. Repeat M times
    5. Extract answers, majority vote
    """
    prompt = build_prompt(problem)
    candidates = [prompt] * config.num_samples

    # M rounds of generate -> execute -> continue
    for step in range(config.num_generations):
        candidates = _generate_vllm(candidates)

        # Execute code blocks where model stopped at ```output
        new_candidates = []
        any_code_run = False
        for text in candidates:
            if text.rstrip().endswith("```output"):
                code = extract_last_code_block(text)
                if code:
                    success, result = executor.execute(code)
                    text += f"\n{result}\n```\n"
                    any_code_run = True
                else:
                    text += "\n\n```\n"
            new_candidates.append(text)
        candidates = new_candidates

        # Early exit if no candidate requested code execution this round
        if not any_code_run:
            break

    # Extract answers from all candidates
    answers = []
    for text in candidates:
        boxed = extract_boxed(text)
        if boxed:
            val = to_int(boxed)
            if 0 <= val <= 99999:
                answers.append(val)

    if not answers:
        return 0

    winner = Counter(answers).most_common(1)[0][0]
    print(f"  -> {len(answers)}/{config.num_samples} answers extracted, winner={winner}")
    return winner


# Quick test
print("Testing solve on a simple problem...")
test_answer = solve("What is 17 * 23?")
print(f"Result: 17*23 = {test_answer} (expected 391)")

In [ ]:
# =============================================
# SUBMISSION / LOCAL EVALUATION
# =============================================

if IS_SUBMISSION:
    # --- Kaggle Competition Mode ---
    # The evaluation API sends problems one at a time.
    # We solve each and return the answer.
    import kaggle_evaluation.aimo_3_inference_server
    
    def predict(problem_df: pd.DataFrame) -> pd.DataFrame:
        problem = problem_df["problem"].iloc[0]
        answer = solve(problem)
        return pd.DataFrame({"id": problem_df["id"], "answer": [answer]})
    
    server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)
    server.serve()

else:
    # --- Local Testing Mode ---
    # Test on the 10 reference problems to verify everything works.
    ref_path = "/kaggle/input/ai-mathematical-olympiad-progress-prize-3/reference.csv"
    if not os.path.exists(ref_path):
        ref_path = "../data/reference.csv"  # Local fallback
    
    test_df = pd.read_csv(ref_path)
    print(f"Testing on {len(test_df)} reference problems...\n")
    
    correct = 0
    for idx, row in test_df.iterrows():
        pred = solve(row["problem"])
        true = row["answer"]
        is_ok = pred == true
        correct += is_ok
        print(f"#{idx+1}: pred={pred}, true={true} {'OK' if is_ok else 'WRONG'}")
    
    print(f"\nResult: {correct}/{len(test_df)} = {correct/len(test_df):.0%}")